# 10 minutes to MAGL

A short introduction to `limnotech_rating_curves`, which fits stage-discharge rating
curves, compares candidate models and maps the results for the MAGL sensor network.

Stage is in feet and discharge in cfs throughout. Run it on the `rating_curves`
environment.

The package reads `data/`, `lrc_cache/` and `output/` relative to the working
directory, and does so at import, so start from the repository root:

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "examples":
    os.chdir("..")

import limnotech_rating_curves as lrc

Path.cwd(), lrc.__version__

The MAGL loaders read from `data/`, which holds the registry and the field records:

```
data/
├── magl/
│   ├── magl_discharge.csv                                       every manual discharge measurement, with its timestamp
│   ├── magl_discharge_report.csv                                the reviewed measurements, with field-recorded stage
│   ├── magl_station_control_point_elevations.csv                the control-point survey
│   ├── magl_changes_to_station_control_point_elevations.csv     sensor moves, and when they happened
│   ├── geolux_sensors.csv                                       which stations are Geolux
│   └── sites/
│       ├── clusters.csv                                         cluster names
│       ├── colocated_pairs.csv                                  station and the USGS gage beside it
│       ├── standalone_gages.csv                                 gages with no co-located station
│       └── special_stations.csv                                 stations read from the pickle below
└── timeseries/
    └── magl_spreadsheet_timeseries.pkl                          the scraped spreadsheet stage record
```

Everything under `magl/` is in the repository. 

The `timeseries/magl_spreadsheet_timeseries.pkl` is the results from running the extract step of `LimnoTech/magl_backfill`, and is not included in the repository.
It is important for making sure a control-point offset can be measured. 

In a `.env` file, you can specify `LRC_DATA_DIR` to points elsewhere.
See [`data/magl/sites/`](../data/magl/sites/README.md) for the registry
files and [`settings.py`](../src/limnotech_rating_curves/settings.py) for every path.

## Loading measurements

See the [`data`](../src/limnotech_rating_curves/data/README.md) README.

Fetching the field measurements at a USGS gage:

In [ ]:
gage = lrc.gage_sample("04176356", start="2016-01-01")
gage

The measured pairs, as a DataFrame:

In [ ]:
gage.to_frame().head()

A sample also records where it came from and what it covers:

In [ ]:
len(gage), gage.site_id, gage.source, gage.stage_label, gage.stage_range

Reading a MAGL sensor, whose manual discharge measurements are matched to its stage
record:

In [ ]:
sbr09 = lrc.sensor_sample("SBR-09")
sbr09.to_frame()

A MAGL sensor measures distance down to the water surface, so `sensor_sample`
references it to a datum first. The default, `stage_datum="lowest"`, takes gage height
from the sensor's own lowest reading; pass a surveyed elevation instead where there is
one.

Building a sample from your own arrays, or from a DataFrame with stage and discharge
columns:

In [ ]:
from limnotech_rating_curves import Sample

mine = Sample.of([1.2, 2.4, 3.1, 4.0], [4.0, 22.5, 61.0, 118.0],
                 site_id="demo", source="field_notes")
mine

## Fitting a rating

Fitting the default model, a single-segment power law:

In [ ]:
rating = lrc.fit_rating(gage)          # single-segment power law, sampled with NUTS
rating.plot();

The estimator is the USGS [`ratingcurve`](https://thodson-usgs.github.io/ratingcurve/meta/intro.html) package, which supplies
the power-law and spline families. See [Background](https://thodson-usgs.github.io/ratingcurve/meta/background.html) for the
equation and [Implementation](https://thodson-usgs.github.io/ratingcurve/meta/implementation.html) for the priors and the two
samplers. `method="advi"` fits by variational inference instead of NUTS.

Predicting discharge at a stage, with its 95% credible interval:

In [ ]:
rating.predict(5.0), rating.interval(5.0)

Neither extrapolates: outside the stage range the fitted curve covers, both return NaN.

Tabulating the rating on an even stage grid, for a lookup table:

In [ ]:
rating.table(step=0.1).head()

`table` runs from just above the fitted stage of zero flow to the top of the
measurements, and `stage_max=` extends it further.

The stage of zero flow is *e* in *Q = C(h − e)^β*, the parameter a short record
constrains worst. The power law centers a prior on Johnson's three-point estimate of
it; `zero_flow=` takes a number, `"johnson"`, or `"infer"`.

In [ ]:
rating.zero_flow, lrc.estimate_zero_flow(gage)

## Comparing models

Fitting several models to the same measurements:

In [ ]:
comparison = lrc.compare(gage, models=["power_law", "power_law_2seg", "spline"])
comparison.metrics

Ranking them by predictive score. `elpd_diff` is each model's expected out-of-sample
predictive density minus the best one's, and `dse` is the standard error of that
difference; a model within about two `dse` of the top is not distinguishable from it on
this sample.

In [ ]:
comparison.ranking()

The best-scoring model, and the reason for any model that did not fit:

In [ ]:
comparison.best.name, comparison.failures

Every fitted curve on one axis:

In [ ]:
comparison.plot();

Models are named by key, and `models=` also takes the group names `"all"`, `"default"`,
`"ratingcurve"`, `"bdrc"` and `"spreadsheet_forms"`.

| key | model | backend |
| --- | --- | --- |
| `power_law` | power law, one segment | [ratingcurve](https://thodson-usgs.github.io/ratingcurve/notebooks/getting-started-tutorial.html) |
| `power_law_2seg`, `power_law_3seg` | segmented power law | [ratingcurve](https://thodson-usgs.github.io/ratingcurve/notebooks/segmented-power-law-tutorial.html) |
| `spline` | natural spline in log space | [ratingcurve](https://thodson-usgs.github.io/ratingcurve/notebooks/spline-tutorial.html) |
| `bdrc_gplm0`, `bdrc_gplm`, `bdrc_plm0`, `bdrc_plm` | generalized power law | bdrc, ported from R |
| `linear`, `quadratic`, `exponential` | least-squares trendlines | this package |

A channel with a floodplain, or with a control that drowns out at high flow, does not
stay on a single log-log line; the segmented models estimate a breakpoint where the
slope changes. See the
[segmented power law](https://thodson-usgs.github.io/ratingcurve/notebooks/segmented-power-law-tutorial.html) and
[model selection](https://thodson-usgs.github.io/ratingcurve/notebooks/model-selection-tutorial.html) tutorials.

The last three keys are the trendline forms the MAGL field workbooks use, refitted by
least squares rather than read from the workbook's coefficients, so a workbook's curve
can be scored against the Bayesian fits.

## Checking a fit

Whether the sampler converged:

In [ ]:
print(lrc.convergence_report(rating))

`lrc.convergence(rating)` gives R-hat and effective sample size per parameter. See the
ratingcurve [troubleshooting page](https://thodson-usgs.github.io/ratingcurve/meta/troubleshooting.html) for what to change
when a fit does not converge.

Residuals against stage, which show whether a form is off over part of the range:

In [ ]:
rating.plot_residuals();

Cross-validation refits on subsets and scores the measurements each fold held out.
Small samples get leave-one-out, larger ones stage-stratified holdout splits, and every
model is scored on the same folds:

In [ ]:
cv = lrc.cross_validate(sbr09, models=["power_law"])
cv.headline

## Sites

A `SiteRating` holds one site's sample, its coordinates and every fit made there:

In [ ]:
site = lrc.SiteRating(sample_id="usgs_gage:04176356", source="usgs_gage",
                      label="USGS 04176356", sample=gage,
                      coords=(42.16, -83.84), gage="04176356")
site.attach_fit(rating.result)

[(fit.key, fit.n, fit.status) for fit in site.successful_fits]

`sample_id` is `"<source>:<site>"` rather than the bare site id because one sensor can
produce more than one rating. SBR-01's stage against MAGL discharge and the same stage
against the co-located USGS gagings are separate ratings, and both would otherwise be
keyed to SBR-01.

The MAGL registry under `data/magl/sites/` builds sites for you. Listing the sensors
that have discharge measurements:

In [ ]:
from limnotech_rating_curves.data import magl

magl.list_sensors()

Assembling one cluster's sites, with samples and coordinates filled in:

In [ ]:
sites = magl.assemble_magl_sites(sites=["SBR"], sources=["magl", "usgs_gage"],
                                 min_points=3, years=5)

[(s.sample_id, len(s.sample), s.coords) for s in sites]

- `sites=` takes a sensor (`"SBR-09"`), a cluster (`"SBR"`) or a gage number; `None`
  takes everything, and only what is selected is fetched.
- `sources=` selects `"magl"` (sensor stage against MAGL discharge), `"colocated"`
  (sensor stage against USGS gagings) or `"usgs_gage"` (the gage alone).
- `years=` bounds how far back a gage's field measurements are taken. Channels change,
  so an old gaging describes a rating that no longer applies.
- `on_vpn=False` skips the pagaia client and takes station positions from
  `magl_station_coordinates.csv`, shipped with the package. Left at `True`, the
  database is asked first and the CSV fills in what it does not answer for.

Fitting each site in turn:

In [ ]:
from limnotech_rating_curves.models import catalog

entry = catalog.get("power_law")

for s in sites:
    fit = entry.fit(s.sample)
    s.attach_fit(fit)
    print(f"{s.sample_id:20s} n={fit.n:3d} {fit.status} {fit.reason}")

`entry.fit` returns a `Fit` whether or not the model could be fitted, with `fit.ok` and
`fit.reason`, so one failure does not stop the loop. `lrc.fit_rating` raises instead.
`attach_fit` replaces an earlier fit of the same model rather than duplicating it.

## Pooling short records

A single-segment power law has three parameters in the mean, so three measurements fit
it exactly and leave nothing to estimate the scatter from, and two cannot fit it at
all. Most MAGL sensors have between three and seven.

`fit_hierarchical` fits every site in one model, drawing each site's scatter from a
population estimated in the same fit: sites with long records set the population, and
short ones borrow from it. Level and stage of zero flow stay local to each site.

In [ ]:
joint = lrc.fit_hierarchical([s.sample for s in sites])

print(joint.summary())

The co-located USGS gages are added by default (`reference=True`); they carry the long
records that determine the population.

The last line reads `usable` only with no divergences and every R-hat inside tolerance.
A few divergences are common with this little data per site, and a longer warm-up with
a higher target acceptance rate usually clears them:

In [ ]:
joint = lrc.fit_hierarchical([s.sample for s in sites],
                             tune=6000, target_accept=0.995)

joint.converged

Each site's scatter, beside the population it was drawn from. A three-measurement site
now has one, between what its own points show and what its cluster does:

In [ ]:
joint.sigma()

Filing each site's marginal of the joint posterior onto `site.fits`, where the map and
the score tables read it like any other fit:

In [ ]:
for s in sites:
    s.attach_marginal_fit(joint)

[(s.sample_id, [f.key for f in s.fits]) for s in sites]

## The map

Writing the interactive map, one self-contained HTML file with the sites, their
measurements, every curve fitted at them and the scores that compare those curves:

In [ ]:
from limnotech_rating_curves.export.mapview import build_map

build_map(sites, output_html="output/magl_map.html",
          title="MAGL power-law ratings. Hover a site to see its fits.",
          exports=False)

Sites with no coordinates, or with no successful fit, are left off it but stay in
`sites`. `exports=True` also writes each fit's manifest and posterior as a zip beside
the HTML.

## Saving and loading

Saving every successful fit at every site, then reading them back. A saved rating is a
manifest (JSON) plus its posterior (NetCDF); the manifest is what makes it loadable,
since a bare `.nc` records draws over parameter names and nothing about the site or the
stage axis they belong to:

In [ ]:
for s in sites:
    lrc.save_site(s, "output/fitted_curves")

saved = lrc.load_ratings("output/fitted_curves")
list(saved)[:3]

A `SavedRating` answers `.predict`, `.table`, `.curve` and `.plot` with the same
arguments as a live fit, so reporting code runs unchanged against stored ratings.

## Where to go next

Each subpackage has a README:

- [`data/`](../src/limnotech_rating_curves/data/README.md) — loaders, stage datums,
  stage of zero flow
- [`models/`](../src/limnotech_rating_curves/models/README.md) — the catalog and the
  estimator backends
- [`model_selection/`](../src/limnotech_rating_curves/model_selection/README.md) —
  cross-validation, predictive scores, convergence
- [`export/`](../src/limnotech_rating_curves/export/README.md) — figures, the map,
  saving and reloading
- [`settings.py`](../src/limnotech_rating_curves/settings.py) — every tunable value
  with its default, each overridable from the environment as `LRC_` + its name; start
  from [`.env.example`](../.env.example)

For the estimator underneath:

- [ratingcurve documentation](https://thodson-usgs.github.io/ratingcurve/meta/intro.html) — [installation](https://thodson-usgs.github.io/ratingcurve/meta/installation.html),
  [getting started](https://thodson-usgs.github.io/ratingcurve/notebooks/getting-started-tutorial.html),
  [tutorials](https://thodson-usgs.github.io/ratingcurve/meta/tutorials.html), [API](https://thodson-usgs.github.io/ratingcurve/api.html)
- [thodson-usgs/ratingcurve](https://github.com/thodson-usgs/ratingcurve) — the source
- Hodson et al. (2024), *Ratingcurve: A Python Package for Fitting Streamflow Rating
  Curves*, Hydrology 11(2),
  [doi:10.3390/hydrology11020014](https://doi.org/10.3390/hydrology11020014)